In [1]:
import pandas as pd

energy = pd.read_csv("C:/Users/yessine/ai_journey/projects/stage/data/train.csv")
buildings = pd.read_csv("C:/Users/yessine/ai_journey/projects/stage/data/building_metadata.csv")

print(energy.columns)
print(buildings.columns)

Index(['building_id', 'meter', 'timestamp', 'meter_reading'], dtype='object')
Index(['site_id', 'building_id', 'primary_use', 'square_feet', 'year_built',
       'floor_count'],
      dtype='object')


In [2]:
buildings.columns

Index(['site_id', 'building_id', 'primary_use', 'square_feet', 'year_built',
       'floor_count'],
      dtype='object')

In [3]:
energy.columns

Index(['building_id', 'meter', 'timestamp', 'meter_reading'], dtype='object')

In [4]:
energy["meter"].unique()

array([0, 3, 1, 2], dtype=int64)

In [5]:
energy["meter"].value_counts().reset_index()

,meter,count
0,0,12060910
1,1,4182440
2,2,2708713
3,3,1264037


In [6]:
energy["date"] = pd.to_datetime(energy["timestamp"]).dt.floor("D")

In [7]:
energy["month_num"]=energy["date"].dt.to_period("M")

In [8]:
energy["year"]=energy["date"].dt.to_period("Y")

In [9]:
eb=energy.merge(buildings,on='building_id',how="left")

In [10]:
eb.shape

(20216100, 12)

In [11]:
eb.isnull().sum()

building_id             0
meter                   0
timestamp               0
meter_reading           0
date                    0
month_num               0
year                    0
site_id                 0
primary_use             0
square_feet             0
year_built       12127645
floor_count      16709167
dtype: int64

In [12]:
buildings.isna().sum()

site_id           0
building_id       0
primary_use       0
square_feet       0
year_built      774
floor_count    1094
dtype: int64

In [13]:
buildings.drop(columns=["year_built", "floor_count"], inplace=True)

In [14]:
eb.drop(columns=["year_built", "floor_count"], inplace=True)

In [15]:
print(eb.columns)

Index(['building_id', 'meter', 'timestamp', 'meter_reading', 'date',
       'month_num', 'year', 'site_id', 'primary_use', 'square_feet'],
      dtype='object')


In [16]:
buildings.columns

Index(['site_id', 'building_id', 'primary_use', 'square_feet'], dtype='object')

In [17]:
facilities_df=buildings

In [18]:
facilities_df.rename(columns={"building_id":"facility_id"},inplace=True)

In [19]:
facilities_df.drop_duplicates()

,site_id,facility_id,primary_use,square_feet
0,0,0,Education,7432
1,0,1,Education,2720
2,0,2,Education,5376
3,0,3,Education,23685
4,0,4,Education,116607
...,...,...,...,...
1444,15,1444,Entertainment/public assembly,19619
1445,15,1445,Education,4298
1446,15,1446,Entertainment/public assembly,11265
1447,15,1447,Lodging/residential,29775


In [20]:
facilities_df.shape

(1449, 4)

In [21]:
subset = energy[energy["building_id"].isin([0, 1])]

In [22]:
subset.shape

(17568, 7)

In [23]:
pivot_df = subset.pivot_table(
    index=["building_id", "date"],
    columns="meter",
    values="meter_reading",
    aggfunc="sum"
)

In [24]:
pivot_df.head()

meter                     0
building_id date           
0           2016-01-01  0.0
            2016-01-02  0.0
            2016-01-03  0.0
            2016-01-04  0.0
            2016-01-05  0.0

In [25]:
pivot_df.shape

(732, 1)

In [26]:
pivot_df = pivot_df.reset_index()

In [27]:
pivot_df.columns

Index(['building_id', 'date', 0], dtype='object', name='meter')

In [28]:
energy.columns

Index(['building_id', 'meter', 'timestamp', 'meter_reading', 'date',
       'month_num', 'year'],
      dtype='object')

In [29]:
ener=energy.groupby("building_id").agg(
    total_meter=('meter','nunique')
)

In [30]:
ener.sort_values("total_meter",ascending=False).head(10)

,total_meter
building_id,
1258,4
1296,4
1331,4
1298,4
1297,4
1232,4
1249,4
1301,4
1295,4


In [31]:
b1258 = energy[energy["building_id"] == 1258]

In [32]:
b1258["meter"].value_counts()

meter
0    8784
1    8784
2    8770
3    5629
Name: count, dtype: int64

In [33]:
pivot_test = b1258.pivot_table(
    index=["building_id", "date"],
    columns="meter",
    values="meter_reading",
    aggfunc="sum"
)

In [34]:
pivot_test = pivot_test.reset_index()

In [35]:
pivot_test.shape

(366, 6)

In [36]:
pivot_test.head()

meter,building_id,date,0,1,2,3
0,1258,2016-01-01,22384.966,81701.65,182834.10,6726.3170
1,1258,2016-01-02,22712.064,84872.48,240597.82,46980.0500
2,1258,2016-01-03,22944.321,83892.12,217865.14,40438.3610
3,1258,2016-01-04,22858.776,96508.46,252979.45,22543.2661
4,1258,2016-01-05,23823.994,102095.87,269996.08,95499.1800


In [37]:
pivot_test = pivot_test.rename(columns={
    0: "electricity_kwh",
    1: "chilled_water_kwh",
    2: "steam_kwh",
    3: "hot_water_kwh"
})

In [38]:
pivot_test.columns

Index(['building_id', 'date', 'electricity_kwh', 'chilled_water_kwh',
       'steam_kwh', 'hot_water_kwh'],
      dtype='object', name='meter')

In [39]:
energy["date"].head()


0   2016-01-01
1   2016-01-01
2   2016-01-01
3   2016-01-01
4   2016-01-01
Name: date, dtype: datetime64[ns]

In [40]:
energy_pivot = energy.pivot_table(
    index=["building_id", "date"],
    columns="meter",
    values="meter_reading",
    aggfunc="sum"
)

In [41]:
energy_pivot = energy_pivot.reset_index()

In [42]:
energy_pivot = energy_pivot.rename(columns={
    0: "electricity_kwh",
    1: "chilled_water_kwh",
    2: "steam_kwh",
    3: "hot_water_kwh",
    "building_id": "facility_id"
})

In [43]:
energy_pivot = energy_pivot.fillna(0)

In [44]:
energy_pivot.shape

(517360, 6)

In [45]:
energy["meter"].value_counts()

meter
0    12060910
1     4182440
2     2708713
3     1264037
Name: count, dtype: int64

In [46]:
energy_pivot.columns

Index(['facility_id', 'date', 'electricity_kwh', 'chilled_water_kwh',
       'steam_kwh', 'hot_water_kwh'],
      dtype='object', name='meter')

In [47]:
print((energy_pivot["chilled_water_kwh"] > 0).sum())
print((energy_pivot["steam_kwh"] > 0).sum())
print((energy_pivot["hot_water_kwh"] > 0).sum())

156206
107582
44215


In [48]:
print(energy_pivot["chilled_water_kwh"].isna().sum())

0


In [49]:
energy.groupby("meter")["building_id"].nunique()

meter
0    1413
1     498
2     324
3     145
Name: building_id, dtype: int64